### Imports

In [115]:
import sklearn as sk
import gensim as gsn
import numpy as np
import pandas as pd
import os
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation as LDA
from sklearn.model_selection import train_test_split
import pyLDAvis as vis

### Load & split data

In [116]:
path = os.getcwd()
df = pd.read_csv(path + "/data/processed_data.csv")
partis_corres = pd.read_csv(path + "/data/partis_correspondance.csv", sep=";")

In [117]:
partis_corres

,parti,groupe,Unnamed: 2
0,non mentionné,Indépendant/micro-partis,NaN
1,Parti communiste français,Extrême-gauche,NaN
2,Parti socialiste,Gauche,NaN
3,Rassemblement pour la République,Droite gaulliste,NaN
4,Union pour la démocratie française,Droite libérale,NaN
...,...,...,...
903,Mouvement d'écologie spirituelle,Indépendant/micro-partis,NaN
904,Ecologie occitane,Indépendant/micro-partis,NaN
905,Aix écologie,Indépendant/micro-partis,NaN
906,Institut de recherche socialiste,Indépendant/micro-partis,NaN


In [118]:
df = pd.merge(df, partis_corres, left_on='titulaire-soutien', right_on='parti')

In [119]:
df_train, df_test = train_test_split(df, test_size=0.2, train_size=0.8)

In [120]:
df

,id,text,date,contexte-election,contexte-tour,cote,departement,departement-insee,identifiant de circonscription,titulaire-sexe,...,titulaire-mandat-en-cours,titulaire-mandat-passe,titulaire-associations,titulaire-autres-statuts,titulaire-soutien,titulaire-liste,titulaire-decorations,parti,groupe,Unnamed: 2
0,EL065_L_1973_03_001_01_1_PF_03,Sciences Po / fonds CEVIPOF\nREPUBLIQUE FRANÇA...,1973-03-04,législatives,1,EL065,01,01 - Ain,1.0,homme,...,non mentionné,non mentionné,culture,non mentionné,non mentionné,Faîtes confiance aux jeunes d'aujourd'hui,oui,non mentionné,Indépendant/micro-partis,NaN
1,EL065_L_1973_03_001_01_1_PF_04,Sciences Po / fonds CEVIPOF\nREPUBLIQUE FRANÇA...,1973-03-04,législatives,1,EL065,01,01 - Ain,1.0,homme,...,non mentionné,non mentionné,politique,résistant,Parti communiste français,Union populaire et victoire du programme commun,non,Parti communiste français,Extrême-gauche,NaN
2,EL065_L_1973_03_001_01_1_PF_05,Sciences Po / fonds CEVIPOF\nREPUBLIQUE FRANÇA...,1973-03-04,législatives,1,EL065,01,01 - Ain,1.0,homme,...,maire;conseiller général,non mentionné,non mentionné,réserve;combattant,non mentionné,Majorité Ve République,oui,non mentionné,Indépendant/micro-partis,NaN
3,EL065_L_1973_03_001_01_1_PF_07,Sciences Po / fonds CEVIPOF\nElections Législa...,1973-03-04,législatives,1,EL065,01,01 - Ain,1.0,homme,...,non mentionné,non mentionné,syndicat;groupes de pression;politique,non mentionné,Parti socialiste unifié,En finir avec la société actuelle,non,Parti socialiste unifié,Gauche,NaN
4,EL065_L_1973_03_001_02_1_PF_02,REPUBLIQUE FRANÇAISE - LIBERTÉ - EGALITÉ - FRA...,1973-03-04,législatives,1,EL065,01,01 - Ain,2.0,homme,...,non mentionné,non mentionné,non mentionné,non mentionné,Parti communiste français,Union populaire et victoire du programme commun,non,Parti communiste français,Extrême-gauche,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16201,EL198_L_1993_03_095_09_1_PF_08,Sciences Po / fonds CEVIPOF\nMarcel PORCHER\nl...,1993-03-21,législatives,1,EL198,95,95 - Val-d'Oise,9,homme,...,maire-adjoint,non mentionné,non mentionné,non mentionné,Rassemblement pour la République,non mentionné,non,Rassemblement pour la République,Droite gaulliste,NaN
16202,EL198_L_1993_03_095_09_1_PF_09,Sciences Po / fonds CEVIPOF\nSTOP\nA\nL'IMMIGR...,1993-03-21,législatives,1,EL198,95,95 - Val-d'Oise,9,homme,...,non mentionné,non mentionné,non mentionné,non mentionné,Trop d'immigrés la France aux Français,non mentionné,non,Trop d'immigrés la France aux Français,Indépendant/micro-partis,NaN
16203,EL198_L_1993_03_095_09_1_PF_10,Sciences Po / fonds CEVIPOF\nElections Législa...,1993-03-21,législatives,1,EL198,95,95 - Val-d'Oise,9,homme,...,non mentionné,député sortant,non mentionné,non mentionné,non mentionné,non mentionné,non,non mentionné,Indépendant/micro-partis,NaN
16204,EL198_L_1993_03_095_09_1_PF_12,Sciences Po / fonds CEVIPOF\nRÉPUBLIQUE FRANÇA...,1993-03-21,législatives,1,EL198,95,95 - Val-d'Oise,9,non déterminé,...,non mentionné,non mentionné,non mentionné,non mentionné,Nouveaux écologistes du rassemblement nature e...,non mentionné,non,Nouveaux écologistes du rassemblement nature e...,Ecologiste,NaN


### Count Vectorizer

In [121]:
stopwords = [x.strip() for x in open('data/stop_words.txt').readlines()]
vectorizer = CountVectorizer(max_features=200, max_df=0.95, min_df=2, stop_words=stopwords)
vectorizer.fit(df_train['text'].to_list())
df_train_counts = vectorizer.transform(df_train['text'])
df_test_counts = vectorizer.transform(df_test['text'])

vocab = vectorizer.vocabulary_.keys()

c:\Users\Utilisateur\Documents\ENSAE\3A\NLP\NLP\venv\Lib\site-packages\sklearn\feature_extraction\text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['allã', 'anmoins', 'aoã', 'aprã', 'but', 'celã', 'cembre', 'chã', 'cinquantiã', 'cinquiã', 'delã', 'derriã', 'deuxiã', 'diffã', 'dixiã', 'douziã', 'dã', 'exceptã', 'eã', 'faã', 'fã', 'gislatives', 'holã', 'huitiã', 'hã', 'lections', 'lã', 'majoritã', 'malgrã', 'mement', 'mãªme', 'mãªmes', 'neuviã', 'nommã', 'nã', 'ohã', 'ollã', 'olã', 'onziã', 'oã¹', 'particuliã', 'passã', 'piã', 'plutã', 'premiã', 'prã', 'putã', 'quatriã', 'quelqu', 're', 'rement', 'rent', 'rente', 'rentes', 'rents', 'res', 'revoilã', 'septiã', 'sixiã', 'sormais', 'taient', 'tais', 'tait', 'tat', 'tiez', 'tions', 'tre', 'troisiã', 'trã', 'tã', 'voilã', 'votã', 'vrier', 'vã', 'ãªtes', 'ãªtre', 'ãªtreãªtre'] not in stop_words.
  warnings.warn(


In [122]:
train_split = df_train['text'].apply(lambda x: x.split())
doc_lengths = train_split.apply(lambda x: len(x)).to_list()

### LDA

In [123]:
lda = LDA(n_components=10, max_iter=5, random_state=0)
lda.fit(df_train_counts)
topic_term_dists = lda.components_

In [124]:
doc_topic_dists = lda.transform(df_train_counts)

### Visualization

In [125]:
term_frequency = np.array(np.sum(df_train_counts, axis =0)).flatten()

In [126]:
lda_data = {'topic_term_dists': topic_term_dists,
            'doc_topic_dists' : doc_topic_dists,
            'doc_lengths' : doc_lengths,
             'vocab' : vocab,
             'term_frequency' : term_frequency}


lda_viz = vis.prepare(**lda_data)


In [127]:
vis.display(lda_viz)